In [1]:
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.


### One Region

In [2]:
import duckdb

# Point exactly to the file shown in your sidebar
base_path = "/mnt/shared_data/finflow/gfw_raw/2025/SEAFDEC.parquet"

# Now the query will work
count = duckdb.query(f"SELECT count(*) FROM '{base_path}'").fetchone()[0]
print(f"Total points captured for SEAFDEC 2025: {count:,}")

# Look at the data
duckdb.query(f"SELECT * FROM '{base_path}' LIMIT 5").show()

Total points captured for SEAFDEC 2025: 4,556,076
┌────────┬────────┬────────┐
│  lat   │  lon   │ hours  │
│ double │ double │ double │
├────────┼────────┼────────┤
│ -28.36 │ 109.72 │    1.0 │
│ -28.35 │ 109.52 │    1.0 │
│ -28.34 │ 109.64 │    2.0 │
│ -28.34 │ 109.68 │    1.0 │
│ -28.34 │  109.7 │    1.0 │
└────────┴────────┴────────┘



In [3]:
duckdb.query(f"""
    SELECT 
        round(lat, 1) as lat_bin, 
        round(lon, 1) as lon_bin, 
        sum(hours) as activity
    FROM '{base_path}'
    GROUP BY 1, 2
    ORDER BY activity DESC
    LIMIT 10
""").show()

┌─────────┬─────────┬──────────┐
│ lat_bin │ lon_bin │ activity │
│ double  │ double  │  double  │
├─────────┼─────────┼──────────┤
│    -7.2 │   112.7 │ 394616.0 │
│    -6.1 │   106.9 │ 340870.0 │
│    19.3 │   105.8 │ 256376.0 │
│     1.3 │   103.9 │ 225593.0 │
│     1.3 │   104.0 │ 216958.0 │
│    13.1 │   100.9 │ 214950.0 │
│     2.8 │   101.3 │ 191308.0 │
│    10.7 │   106.8 │ 172578.0 │
│    14.6 │   121.0 │ 164800.0 │
│    13.8 │   109.3 │ 164370.0 │
└─────────┴─────────┴──────────┘
  10 rows            3 columns



### Currently Downloaded

In [4]:
import duckdb

global_path = "/mnt/shared_data/finflow/gfw_raw/2025/*.parquet"

# This query aggregates data and shows you the source filename
stats = duckdb.query(f"""
    SELECT 
        filename, 
        count(*) as total_points,
        round(avg(hours), 2) as avg_activity,
        max(hours) as max_activity
    FROM read_parquet('{global_path}', filename=True)
    GROUP BY filename
""").df()

print(stats)

                                            filename  total_points  \
0  /mnt/shared_data/finflow/gfw_raw/2025/CCAMLR.p...          2421   
1  /mnt/shared_data/finflow/gfw_raw/2025/IPHC.par...        525066   
2  /mnt/shared_data/finflow/gfw_raw/2025/IATTC.pa...       3234526   
3  /mnt/shared_data/finflow/gfw_raw/2025/APFIC.pa...       5029329   
4  /mnt/shared_data/finflow/gfw_raw/2025/SEAFDEC....       4556076   

   avg_activity  max_activity  
0          1.72         421.0  
1          2.71       17836.0  
2          2.26       31820.0  
3         14.90       76567.0  
4          8.44       76478.0  
